In [1]:
# Milestone 5 · Step 1 — project code (commit 7d3c739) + keys from Kaggle Secrets (never printed)
!git clone -q https://github.com/jamalla/arabic-book-recommender-agent.git repo
!pip install -q -r repo/requirements.txt
import os, sys
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = secrets.get_secret("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_PUBLIC_KEY"] = secrets.get_secret("Public Key")  # the label used in Kaggle
os.environ["LANGFUSE_BASE_URL"] = "https://cloud.langfuse.com"
sys.path.insert(0, "repo")

import langchain, langgraph
from langfuse import get_client
print("Langfuse connected:", get_client().auth_check(), "| langchain", langchain.__version__)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
Langfuse connected: True | langchain 1.3.18


In [2]:
# Step 2 — the ONE change in Run B: prompt v1 -> v2 (each new rule targets a failure we measured)
import difflib
from bookrec.agent import SYSTEM_PROMPT_V1, SYSTEM_PROMPT_V2

for line in difflib.unified_diff(SYSTEM_PROMPT_V1.splitlines(), SYSTEM_PROMPT_V2.splitlines(),
                                 "prompt v1", "prompt v2", lineterm=""):
    print(line)

# Step 3 — smoke test before spending a full run: the Mahfouz trap through BOTH agents with prompt v2
from bookrec.agent import BookAgent
from bookrec.agent_langgraph import LangGraphBookAgent

for agent in [BookAgent(prompt_version="v2"), LangGraphBookAgent(prompt_version="v2")]:
    r = agent.answer("رشح لي روايات لنجيب محفوظ", tags=["smoke-test"])
    print(f"\n===== {type(agent).__name__}: {len(r['steps'])} searches {[s['query'] for s in r['steps']]}")
    print(r["answer"])

--- prompt v1
+++ prompt v2
@@ -1,4 +1,11 @@
 You are a friendly bookstore assistant for Arabic readers.
-Use the search_books tool to look up books in our catalog, then recommend books that match
-the reader's request, with one short sentence on why each one fits.
+Rules:
+1. Recommend ONLY books that the search_books tool returned in this conversation. Never recommend
+   a book from your own knowledge, even a famous one, and copy titles exactly as the tool wrote them.
+2. Recommend a book only if its description clearly fits the reader's request. One fitting book is
+   better than several weak ones: never pad the list.
+3. If nothing fits (the book, author or topic is not in our catalog), say so honestly in one sentence
+   and offer to search for something related. Do not list books from memory.
+4. If the results look weak, search again with different words (at most 3 searches).
+5. For each recommended book give the exact title, the author and one short sentence on why it fits.


In [3]:
# Step 4 — Run B: the same plain agent, ONLY the prompt changes (v1 -> v2), the same 12 questions
import pandas as pd
from evals.run_experiment import run

def table(result):
    rows = []
    for r in result.item_results:
        s = {e.name: e.value for e in r.evaluations}
        rows.append({"id": r.item.id.split("-")[-1], "question": r.item.input["question"],
                     "invented_books": s.get("invented_books"), "fit": s.get("fit")})
    return pd.DataFrame(rows).sort_values("id").set_index("id")

def headline(name, t):
    return f"{name}: avg fit {t.fit.mean():.3f} | grounded {(t.invented_books == 0).sum()}/12 | invented books {int(t.invented_books.sum())}"

result_b = run("v2", "openai")
b = table(result_b)
print(headline("Run B (plain + prompt v2)", b))
b

Run B (plain + prompt v2): avg fit 0.917 | grounded 12/12 | invented books 0


,question,invented_books,fit
id,,,
q01,أريد رواية تاريخية,0,1.0
q02,أبحث عن كتاب يساعدني في تربية أطفالي,0,1.0
q03,أريد كتاباً مبسطاً عن الفلسفة,0,1.0
q04,كتاب عن الاقتصاد أو إدارة الأعمال,0,1.0
q05,أريد أن أفهم السياسة في العالم العربي,0,1.0
q06,كتاب علمي عن الطبيعة أو الكون,0,1.0
q07,أحب قراءة السير الذاتية والمذكرات,0,1.0
q08,كتاب عن الفن أو الخط العربي,0,1.0
q09,أريد كتاباً عن السيرة النبوية,0,0.0


In [4]:
# Step 5 — read the evidence for the ONE regression (q09): what did v2 answer, and why did the judge give 0.0?
r09 = next(r for r in result_b.item_results if r.item.id.endswith("q09"))
ev = {e.name: e for e in r09.evaluations}
print("searches:", [s["query"] for s in r09.output["steps"]])
for s in r09.output["steps"]:
    for book in s["books"][:3]:
        print(f"   {book['score']:.2f}  {book['title']}  — {book['category']}")
print("\nanswer:\n" + r09.output["answer"])
print("\njudge:", ev["fit"].value, "-", ev["fit"].comment)

searches: ['كتاب عن السيرة النبوية حياة النبي محمد وأحداث سيرته', 'السيرة النبوية سيرة حياة الرسول محمد تاريخ الدعوة والهجرة والغزوات', 'قصة النبي محمد من المولد والبعثة والهجرة إلى الوفاة السيرة والتاريخ الإسلامي']
   0.27  الأنوار في شمائل النبي المختار (ص)  — الكتب الإسلامية
   0.26  الحياة العسكرية لقائد البشرية صلى الله عليه وسلم  — الكتب الإسلامية
   0.21  في مدينة الرسول صلى الله عليه وسلم  — الكتب الإسلامية
   0.30  الحياة العسكرية لقائد البشرية صلى الله عليه وسلم  — الكتب الإسلامية
   0.27  في مدينة الرسول صلى الله عليه وسلم  — الكتب الإسلامية
   0.19  مطالع المسرات بجلاء دلائل الخيرات - ابيض  — الكتب الإسلامية
   0.19  الأنوار في شمائل النبي المختار (ص)  — الكتب الإسلامية
   0.17  أطلس تاريخ الأنبياء والرسل  — التاريخ والجغرافيا
   0.16  الحياة العسكرية لقائد البشرية صلى الله عليه وسلم  — الكتب الإسلامية

answer:
لا يظهر في نتائج الفهرس كتاب يطابق طلبك عن السيرة النبوية كاملة؛ يمكنني البحث عن كتاب متخصص في شمائل النبي أو غزواته.

judge: 0.0 - The answer recommends nothing des

In [5]:
# Step 6 — Run C: the same prompt v2, ONLY the framework changes (plain OpenAI SDK -> LangGraph), same 12 questions
result_c = run("v2", "langgraph")
c = table(result_c)
print(headline("Run C (LangGraph + prompt v2)", c))
c

Run C (LangGraph + prompt v2): avg fit 1.000 | grounded 12/12 | invented books 0


,question,invented_books,fit
id,,,
q01,أريد رواية تاريخية,0,1.0
q02,أبحث عن كتاب يساعدني في تربية أطفالي,0,1.0
q03,أريد كتاباً مبسطاً عن الفلسفة,0,1.0
q04,كتاب عن الاقتصاد أو إدارة الأعمال,0,1.0
q05,أريد أن أفهم السياسة في العالم العربي,0,1.0
q06,كتاب علمي عن الطبيعة أو الكون,0,1.0
q07,أحب قراءة السير الذاتية والمذكرات,0,1.0
q08,كتاب عن الفن أو الخط العربي,0,1.0
q09,أريد كتاباً عن السيرة النبوية,0,1.0


In [6]:
# Step 7 — B and C differ on ONE question (q09). Real difference or noise? Repeat q09 5x per agent, count refusals.
from evals.evaluators import recommended_titles
from bookrec.agent import BookAgent
from bookrec.agent_langgraph import LangGraphBookAgent

q09 = "أريد كتاباً عن السيرة النبوية"
for agent in [BookAgent(prompt_version="v2"), LangGraphBookAgent(prompt_version="v2")]:
    counts = [len(recommended_titles(agent.answer(q09, tags=["noise-check"])["answer"])) for _ in range(5)]
    refusals = sum(n == 0 for n in counts)
    print(f"{type(agent).__name__:20s} books recommended per run: {counts}  -> refused {refusals}/5")

BookAgent            books recommended per run: [1, 1, 1, 1, 0]  -> refused 1/5
LangGraphBookAgent   books recommended per run: [1, 1, 1, 1, 1]  -> refused 0/5


In [7]:
# Step 8 — the decision table. Run A = official baseline from Milestone 4 (notebooks/04_evaluation.ipynb)
summary = pd.DataFrame([
    {"run": "A  plain + prompt v1 (baseline)", "avg fit": 0.750, "grounded": "11/12", "invented books": 8, "q09 refusals (5 tries)": "-"},
    {"run": "B  plain + prompt v2", "avg fit": round(b.fit.mean(), 3), "grounded": f"{(b.invented_books == 0).sum()}/12",
     "invented books": int(b.invented_books.sum()), "q09 refusals (5 tries)": "1/5"},
    {"run": "C  LangGraph + prompt v2", "avg fit": round(c.fit.mean(), 3), "grounded": f"{(c.invented_books == 0).sum()}/12",
     "invented books": int(c.invented_books.sum()), "q09 refusals (5 tries)": "0/5"},
]).set_index("run")
print("Decision: keep prompt v2 (clear win over v1). LangGraph ties or beats the plain loop on every measure,")
print("so ship LangGraph + prompt v2; the plain agent stays available via AGENT_FRAMEWORK=openai.")
summary

Decision: keep prompt v2 (clear win over v1). LangGraph ties or beats the plain loop on every measure,
so ship LangGraph + prompt v2; the plain agent stays available via AGENT_FRAMEWORK=openai.


,avg fit,grounded,invented books,q09 refusals (5 tries)
run,,,,
A plain + prompt v1 (baseline),0.750,11/12,8,-
B plain + prompt v2,0.917,12/12,0,1/5
C LangGraph + prompt v2,1.000,12/12,0,0/5
